In [ ]:
from modelfacts import ModelFacts
import pandas as pd
from sklearn import metrics
import numpy as np
from scipy.stats import ttest_1samp

In [3]:
df = pd.read_csv("zero_shot_llm_labels.csv")
print(df.shape)
df.head()

(469, 19)


,Unnamed: 0,domain,base_url,text,language,title,Bias,Factual Rating,Credibility Rating,Conspiracy/Pseudoscience,Questionable/Fake,topic,domain_bias,article_bias_gpt_base_prompt,article_bias_gpt_reduced_prompt,article_bias_gpt_bias_only_prompt,article_bias_llama_base_prompt,article_bias_llama_reduced_prompt,article_bias_llama_bias_only_prompt
0,0,foxnews.com,https://www.foxnews.com/us/pentagon-rushed-den...,A Pentagon report found the military was likel...,en,Pentagon rushed to deny service members' vacci...,RIGHT,MIXED,MEDIUM CREDIBILITY,NaN,NaN,religious_exemptions,NaN,RIGHT-CENTER,CENTER,biased,RIGHT-CENTER,CENTER,unbiased
1,1,wilx.com,https://www.wilx.com/2022/02/08/air-force-appr...,WASHINGTON (AP) — The Air Force became the sec...,en,Air Force approves 9 religious exemptions for ...,CENTER,HIGH,HIGH CREDIBILITY,NaN,NaN,religious_exemptions,NaN,CENTER,CENTER,unbiased,RIGHT-CENTER,LEFT,unbiased
2,2,theepochtimes.com,https://www.theepochtimes.com/significant-proo...,‘Significant Proof’ Air Force Has Discriminate...,en,‘Significant Proof’ Air Force Has Discriminate...,RIGHT,MIXED,LOW CREDIBILITY,NaN,True,religious_exemptions,NaN,RIGHT-CENTER,RIGHT,biased,FAR RIGHT,RIGHT,biased
3,3,thegatewaypundit.com,https://www.thegatewaypundit.com/2022/09/tride...,"“Trident Order #12,” disqualified SEALs seekin...",en,"Trident Order #12, Which Disqualified SEALs Fr...",FAR RIGHT,NaN,NaN,NaN,True,religious_exemptions,NaN,FAR RIGHT,RIGHT,biased,LEFT,RIGHT,unbiased
4,4,fox43.com,https://www.fox43.com/article/life/new-novavax...,The Novavax COVID-19 vaccine uses traditional ...,en,New COVID-19 vaccine could override religious ...,CENTER,HIGH,HIGH CREDIBILITY,NaN,NaN,religious_exemptions,NaN,LEFT-CENTER,CENTER,biased,FAR RIGHT,CENTER,unbiased


In [4]:
cols = [
    'article_bias_gpt_base_prompt', 
     'article_bias_llama_base_prompt'
]

# convert to ordinal labels
df[['BiasOrd', 'gptOrd', 'llamaOrd']] = df[
    ['Bias',  'article_bias_gpt_base_prompt', 
     'article_bias_llama_base_prompt']].replace({"FAR LEFT":1, "LEFT":2, "LEFT-CENTER":3,
                             "center  left":3, "CENTER":4,
                                    "RIGHT-CENTER":5, "RIGHT":6, "FAR RIGHT":7})

df['BiasGroup'] = df['Bias'].replace({
    "FAR LEFT":"LEFT", "LEFT-CENTER":"LEFT", 
    "RIGHT-CENTER":"RIGHT", "FAR RIGHT":"RIGHT"})

In [42]:
def mean_signed_difference(y_true, y_pred, alternative):
    diff = np.array(y_true) - np.array(y_pred)
    score = ttest_1samp(list(diff), popmean = 0, alternative = alternative)
    return (np.mean(diff), score.statistic, score.pvalue)

In [46]:
# get stats by topic
st_func = mean_signed_difference
demo_col = 'BiasGroup'
true = 'BiasOrd'


for topic in set(df['topic']):
    print(f"Topic: {topic}")
    for pred in ['gptOrd', 'llamaOrd']:
        data = df[df['topic'] == topic]
        groups = data.groupby(demo_col, observed = False)
        results = []
        for group in groups:
                label = group[0]
                x = group[1]
                if label  == 'LEFT':
                        alternative = 'less'
                elif label == 'RIGHT':
                        alternative = 'greater'
                else:
                        alternative = 'two-sided'
                results.append((label, x[demo_col].count()/len(data)*100, x[demo_col].count(), # % in test data
                        st_func(x[true], x[pred], alternative), 
                        (x[true].mean(), x[true].std()),(x[pred].mean(), x[pred].std()))) # (target mean/std)
        print(f'{pred} stats')
        stats_df = pd.DataFrame(results, 
                columns = ['label', '% in Test Data', 'size', 'Standard Score', 'True: Mean, Std', 'Pred: Mean, Std'])
        stats_df[['mean_diff', 't-stat', 'p-value']] = pd.DataFrame(
            stats_df['Standard Score'].tolist(), 
            index=stats_df.index
        )
        display(stats_df)
    # standard error, mean score, and stdev across  whole data
    print('overall stats: mean diff, t-tstat, p-value, true mean, true std, pred mean, pred std')
    print(st_func(data[true], data[pred], 'two-sided'), (data[true].mean(), data[true].std()), (data[pred].mean(), data[pred].std())) # (target mean/std)

Topic: religious_exemptions
gptOrd stats


,label,% in Test Data,size,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,23.611111,17,"(-0.11764705882352941, -0.5229763603684907, 0....","(4.0, 0.0)","(4.117647058823529, 0.9275204136012635)",-0.117647,-0.522976,6.081577e-01
1,LEFT,9.722222,7,"(-1.5714285714285714, -4.260281680828158, 0.00...","(2.857142857142857, 0.3779644730092273)","(4.428571428571429, 0.9759000729485332)",-1.571429,-4.260282,2.659798e-03
2,RIGHT,66.666667,48,"(0.9791666666666666, 5.76951224640332, 2.99976...","(6.041666666666667, 0.5819385554072728)","(5.0625, 1.2446694853090818)",0.979167,5.769512,2.999765e-07


llamaOrd stats


,label,% in Test Data,size,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,23.611111,17,"(-0.6470588235294118, -1.7338428937260215, 0.1...","(4.0, 0.0)","(4.647058823529412, 1.5387160422974504)",-0.647059,-1.733843,0.102169
1,LEFT,9.722222,7,"(-1.1428571428571428, -1.3333333333333333, 0.1...","(2.857142857142857, 0.3779644730092273)","(4.0, 2.309401076758503)",-1.142857,-1.333333,0.115405
2,RIGHT,66.666667,48,"(1.4791666666666667, 4.758599613937792, 9.4663...","(6.041666666666667, 0.5819385554072728)","(4.5625, 2.030682203784225)",1.479167,4.758600,0.000009


overall stats: mean diff, t-tstat, p-value, true mean, true std, pred mean, pred std
(np.float64(0.7222222222222222), np.float64(2.682901490535651), np.float64(0.009073512486984136)) (np.float64(5.25), np.float64(1.2643542024669252)) (np.float64(4.527777777777778), np.float64(1.935380169534135))
Topic: navy_seals
gptOrd stats


,label,% in Test Data,size,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,13.461538,7,"(-1.4285714285714286, -3.872983346207417, 0.00...","(4.0, 0.0)","(5.428571428571429, 0.9759000729485331)",-1.428571,-3.872983,0.008237
1,LEFT,25.000000,13,"(-1.2307692307692308, -3.5927097051125756, 0.0...","(2.769230769230769, 0.4385290096535146)","(4.0, 1.2909944487358056)",-1.230769,-3.592710,0.001848
2,RIGHT,61.538462,32,"(0.53125, 2.9593201512468634, 0.00293074726952...","(5.9375, 0.6690146823211658)","(5.40625, 1.073414058942533)",0.531250,2.959320,0.002931


... (7 outputs left out)

,label,% in Test Data,size,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,35.555556,16,"(-0.125, -0.6956083436402525, 0.497310545763386)","(4.0, 0.0)","(4.125, 0.7187952884282609)",-0.125000,-0.695608,4.973105e-01
1,LEFT,24.444444,11,"(-0.9090909090909091, -10.000000000000002, 7.9...","(2.909090909090909, 0.30151134457776363)","(3.8181818181818183, 0.4045199174779452)",-0.909091,-10.000000,7.947766e-07
2,RIGHT,40.000000,18,"(0.8888888888888888, 2.4654030685676513, 0.012...","(6.055555555555555, 0.639137490706142)","(5.166666666666667, 1.5434872662825794)",0.888889,2.465403,1.231411e-02


llamaOrd stats


,label,% in Test Data,size,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,35.555556,16,"(0.625, 1.3727425507183655, 0.19000907295096606)","(4.0, 0.0)","(3.375, 1.8211717839530315)",0.625000,1.372743,0.190009
1,LEFT,24.444444,11,"(-0.7272727272727273, -1.3884202690012468, 0.0...","(2.909090909090909, 0.30151134457776363)","(3.6363636363636362, 1.747725795010606)",-0.727273,-1.388420,0.097579
2,RIGHT,40.000000,18,"(1.2222222222222223, 2.6098387456212113, 0.009...","(6.055555555555555, 0.639137490706142)","(4.833333333333333, 2.1213203435596424)",1.222222,2.609839,0.009152


overall stats: mean diff, t-tstat, p-value, true mean, true std, pred mean, pred std
(np.float64(0.5333333333333333), np.float64(1.8032559722947348), np.float64(0.07819845326566348)) (np.float64(4.555555555555555), np.float64(1.3743685418725535)) (np.float64(4.022222222222222), np.float64(2.00554786086551))


In [44]:
# group by just model types, not topics

data = df
for pred in ['gptOrd', 'llamaOrd']:
        groups = data.groupby(demo_col, observed = False)
        results = []
        for group in groups:
                label = group[0]
                x = group[1]
                if label  == 'LEFT':
                        alternative = 'less'
                elif label == 'RIGHT':
                        alternative = 'greater'
                else:
                        alternative = 'two-sided'
                results.append((label, x[demo_col].count()/len(data)*100,  # % in test data
                        st_func(x[true], x[pred], alternative), 
                        (x[true].mean(), x[true].std()),(x[pred].mean(), x[pred].std()))) # (target mean/std)
        print(f'{pred} stats')
        stats_df = pd.DataFrame(results, 
                columns = ['label', '% in Test Data', 'Standard Score', 'True: Mean, Std', 'Pred: Mean, Std'])
        stats_df[['mean_diff', 't-stat', 'p-value']] = pd.DataFrame(
            stats_df['Standard Score'].tolist(), 
            index=stats_df.index
        )
        display(stats_df)

gptOrd stats


,label,% in Test Data,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,37.953092,"(-0.19101123595505617, -1.8083788279627304, 0....","(4.0, 0.0)","(4.191011235955056, 1.4092222841697912)",-0.191011,-1.808379,7.224491e-02
1,LEFT,24.946695,"(-1.7435897435897436, -13.270062178822284, 2.5...","(2.2393162393162394, 0.42850125034164827)","(3.982905982905983, 1.389510618284713)",-1.743590,-13.270062,2.590865e-25
2,RIGHT,37.100213,"(1.2126436781609196, 11.889594121764128, 1.640...","(6.005747126436781, 0.4624279532920934)","(4.793103448275862, 1.3696793518224804)",1.212644,11.889594,1.640805e-24


llamaOrd stats


,label,% in Test Data,Standard Score,"True: Mean, Std","Pred: Mean, Std",mean_diff,t-stat,p-value
0,CENTER,37.953092,"(-0.17415730337078653, -1.1964093444477781, 0....","(4.0, 0.0)","(4.174157303370786, 1.9421013774843439)",-0.174157,-1.196409,2.331370e-01
1,LEFT,24.946695,"(-1.5982905982905984, -8.117745015182635, 2.82...","(2.2393162393162394, 0.42850125034164827)","(3.8376068376068377, 2.067637603131622)",-1.598291,-8.117745,2.821889e-13
2,RIGHT,37.100213,"(1.1551724137931034, 7.154217270568326, 1.1425...","(6.005747126436781, 0.4624279532920934)","(4.850574712643678, 2.11806835126455)",1.155172,7.154217,1.142517e-11
